# Automated Data Cleaning, Analysis & Regression Notebook
**Requirements:** pandas, numpy, matplotlib, scipy, scikit-learn

This notebook performs the following steps:
1. Load dataset
2. Convert currency-like columns to numeric
3. Choose target and predictors
4. Clean missing data
5. Generate descriptive statistics
6. Visualize data
7. Conduct Q-Q plot, confidence intervals, and hypothesis testing
8. Build and evaluate a linear regression model
9. Save results and plots

In [4]:
# requirements: pandas numpy matplotlib scipy scikit-learn
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. load
fn = 'loan.csv'
df = pd.read_csv(fn, encoding='latin1')  # encoding fallback

# quick inspect
print(df.shape)
print(df.columns.tolist())
display(df.head())

# 2. Try to convert currency-like columns to numeric:
def try_numeric(col):
    # remove commas, currency symbols, parentheses; convert to float where possible
    s = df[col].astype(str).str.replace(r'[,$()]','', regex=True).str.replace('--','').str.strip()
    # if contains % maybe remove:
    s = s.str.replace('%','', regex=False)
    # coerce
    num = pd.to_numeric(s, errors='coerce')
    return num

converted = {}
for c in df.columns:
    # Heuristic: if >50% values look numeric when stripped, convert
    sample = df[c].astype(str).head(50).str.replace(r'[,$()%()]','', regex=True).str.strip()
    num_like = sample.apply(lambda x: x.replace('.','',1).lstrip('-').isdigit()).sum()
    if num_like >= 5:  # simple heuristic
        df[c + '_num'] = try_numeric(c)
        converted[c] = c + '_num'

print('Converted columns (candidate numerics):', converted)

# 3. Choose a target and predictors:
numeric_cols = [c for c in df.columns if df[c].dtype in [float, int] or c.endswith('_num')]
print('Numeric columns now:', numeric_cols)

target = numeric_cols[-1]  # change as needed
predictors = [c for c in numeric_cols if c != target][:3]

print('Using target:', target)
print('Using predictors:', predictors)

# 4. Cleaning missing data
for c in numeric_cols:
    if df[c].dtype in [float, int] or c.endswith('_num'):
        df[c] = df[c].fillna(df[c].median())

# 5. Descriptive stats
display(df[numeric_cols].describe().T)

# 6. Visualizations
os.makedirs('/mnt/data/plots', exist_ok=True)
for c in numeric_cols:
    plt.figure()
    df[c].hist(bins=30)
    plt.title(f'Histogram: {c}')
    plt.savefig(f'{c}.png')
    plt.close()

    plt.figure()
    df.boxplot(column=c)
    plt.title(f'Boxplot: {c}')
    plt.savefig(f'{c}.png')
    plt.close()

if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    plt.figure(figsize=(8,6))
    plt.imshow(corr, cmap='coolwarm', interpolation='nearest')
    plt.colorbar()
    plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45)
    plt.yticks(range(len(numeric_cols)), numeric_cols)
    plt.title('Correlation heatmap')
    plt.savefig('/mnt/data/plots/corr_heatmap.png')
    plt.close()
    display(corr)

# 7. Q-Q plot for target
plt.figure()
stats.probplot(df[target], dist='norm', plot=plt)
plt.title(f'Q-Q plot for {target}')
plt.savefig(f'/mnt/data/plots/qq_{target}.png')
plt.close()

# 8. 95% confidence interval for mean of target
n = len(df[target])
mean = df[target].mean()
se = stats.sem(df[target])
ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se) if n>1 else (mean, mean)
print('95% CI for mean:', ci)

# 9. Hypothesis test (two-sided t-test)
pop_value = mean * 0.9
t_stat, p_val = stats.ttest_1samp(df[target], pop_value)
print(f'H0: mean({target}) = {pop_value:.4f}')
print('t-statistic:', t_stat, 'p-value:', p_val)

# 10. Linear regression
if len(predictors) >= 1:
    X = df[predictors].values
    y = df[target].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    print('MSE:', mse, 'RMSE:', rmse, 'R2:', r2)

    plt.figure()
    plt.scatter(y_test, preds)
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Predicted vs Actual')
    plt.savefig('/mnt/data/plots/pred_vs_actual.png')
    plt.close()
else:
    print('Not enough numeric predictors for regression.')

# 11. Save cleaned dataset
df.to_csv('/mnt/data/cleaned_dataset.csv', index=False)
print('Saved cleaned dataset to /mnt/data/cleaned_dataset.csv')
print('Saved plots to /mnt/data/plots/')


(614, 13)
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


Converted columns (candidate numerics): {'Dependents': 'Dependents_num', 'ApplicantIncome': 'ApplicantIncome_num', 'CoapplicantIncome': 'CoapplicantIncome_num', 'LoanAmount': 'LoanAmount_num', 'Loan_Amount_Term': 'Loan_Amount_Term_num', 'Credit_History': 'Credit_History_num'}
Numeric columns now: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Dependents_num', 'ApplicantIncome_num', 'CoapplicantIncome_num', 'LoanAmount_num', 'Loan_Amount_Term_num', 'Credit_History_num']
Using target: Credit_History_num
Using predictors: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']


,count,mean,std,min,25%,50%,75%,max
ApplicantIncome,614.0,5403.459283,6109.041673,150.0,2877.50,3812.5,5795.00,81000.0
CoapplicantIncome,614.0,1621.245798,2926.248369,0.0,0.00,1188.5,2297.25,41667.0
LoanAmount,614.0,145.752443,84.107233,9.0,100.25,128.0,164.75,700.0
Loan_Amount_Term,614.0,342.410423,64.428629,12.0,360.00,360.0,360.00,480.0
Credit_History,614.0,0.855049,0.352339,0.0,1.00,1.0,1.00,1.0
Dependents_num,614.0,0.495114,0.761519,0.0,0.00,0.0,1.00,2.0
ApplicantIncome_num,614.0,5403.459283,6109.041673,150.0,2877.50,3812.5,5795.00,81000.0
CoapplicantIncome_num,614.0,1621.245798,2926.248369,0.0,0.00,1188.5,2297.25,41667.0
LoanAmount_num,614.0,145.752443,84.107233,9.0,100.25,128.0,164.75,700.0
Loan_Amount_Term_num,614.0,342.410423,64.428629,12.0,360.00,360.0,360.00,480.0


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Dependents_num,ApplicantIncome_num,CoapplicantIncome_num,LoanAmount_num,Loan_Amount_Term_num,Credit_History_num
ApplicantIncome,1.000000,-0.116605,0.565181,-0.046531,-0.018615,-0.013777,1.000000,-0.116605,0.565181,-0.046531,-0.018615
CoapplicantIncome,-0.116605,1.000000,0.189218,-0.059383,0.011134,-0.004802,-0.116605,1.000000,0.189218,-0.059383,0.011134
LoanAmount,0.565181,0.189218,1.000000,0.036960,-0.000607,0.049927,0.565181,0.189218,1.000000,0.036960,-0.000607
Loan_Amount_Term,-0.046531,-0.059383,0.036960,1.000000,-0.004705,-0.053623,-0.046531,-0.059383,0.036960,1.000000,-0.004705
Credit_History,-0.018615,0.011134,-0.000607,-0.004705,1.000000,0.012556,-0.018615,0.011134,-0.000607,-0.004705,1.000000
Dependents_num,-0.013777,-0.004802,0.049927,-0.053623,0.012556,1.000000,-0.013777,-0.004802,0.049927,-0.053623,0.012556
ApplicantIncome_num,1.000000,-0.116605,0.565181,-0.046531,-0.018615,-0.013777,1.000000,-0.116605,0.565181,-0.046531,-0.018615
CoapplicantIncome_num,-0.116605,1.000000,0.189218,-0.059383,0.011134,-0.004802,-0.116605,1.000000,0.189218,-0.059383,0.011134
LoanAmount_num,0.565181,0.189218,1.000000,0.036960,-0.000607,0.049927,0.565181,0.189218,1.000000,0.036960,-0.000607
Loan_Amount_Term_num,-0.046531,-0.059383,0.036960,1.000000,-0.004705,-0.053623,-0.046531,-0.059383,0.036960,1.000000,-0.004705


95% CI for mean: (np.float64(0.8271245488524878), np.float64(0.8829731710172192))
H0: mean(Credit_History_num) = 0.7695
t-statistic: 6.013327893899565 p-value: 3.123570176753652e-09
MSE: 0.13100154748936368 RMSE: 0.3619413591859373 R2: -0.0029971720478656483
Saved cleaned dataset to /mnt/data/cleaned_dataset.csv
Saved plots to /mnt/data/plots/
